# **Makemore: Multi-Layer Perceptrons**
Scales the ideas from Micrograd and Makemore: Bigrams into a true multi-layer neural network with a hidden layer and non-linear activation functions. Model moves past and beyond static counting. Introduces data splitting, hyperparameter tuning, under and overfitting, etc. 

**Paper Followed:** A Neural Probabilistic Language Model - Bengio et al. 2003

## **Building Data-Set**

In [2]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
%matplotlib inline

In [3]:
# read in all the words
words = open('names.txt', 'r').read().splitlines()
words[:8]

['emma', 'olivia', 'ava', 'isabella', 'sophia', 'charlotte', 'mia', 'amelia']

In [4]:
# build vocab of characters and mapping to/from integers
chars = sorted(list(set(''.join(words))))
stoi= {s:i+1 for i,s in enumerate(chars)}
stoi['.'] = 0
itos = {i:s for s,i in stoi.items()}
print(itos)

{1: 'a', 2: 'b', 3: 'c', 4: 'd', 5: 'e', 6: 'f', 7: 'g', 8: 'h', 9: 'i', 10: 'j', 11: 'k', 12: 'l', 13: 'm', 14: 'n', 15: 'o', 16: 'p', 17: 'q', 18: 'r', 19: 's', 20: 't', 21: 'u', 22: 'v', 23: 'w', 24: 'x', 25: 'y', 26: 'z', 0: '.'}


In [9]:
block_size = 3 # context length: how many chars do we take to predict the next one
X, Y = [], []
for w in words[:5]:

    print(w)
    context = [0] * block_size
    for ch in w + '.':
        ix = stoi[ch]
        X.append(context)
        Y.append(ix)
        print(''.join(itos[i] for i in context), '--->', itos[ix])
        context = context[1:] + [ix]

X = torch.tensor(X)
Y = torch.tensor(Y)

emma
... ---> e
..e ---> m
.em ---> m
emm ---> a
mma ---> .
olivia
... ---> o
..o ---> l
.ol ---> i
oli ---> v
liv ---> i
ivi ---> a
via ---> .
ava
... ---> a
..a ---> v
.av ---> a
ava ---> .
isabella
... ---> i
..i ---> s
.is ---> a
isa ---> b
sab ---> e
abe ---> l
bel ---> l
ell ---> a
lla ---> .
sophia
... ---> s
..s ---> o
.so ---> p
sop ---> h
oph ---> i
phi ---> a
hia ---> .


## **Embedding Look-Up Table**
Like the Weight matrix

In [5]:
# each 27 chars will have 2 dimensional embedding
# gives each character a (x, y) coordinate pair
C = torch.randn((27, 2))

In [6]:
C[5]

# Under the hood:
# F.one_hot(torch.tensor(5), num_classes=27).float() @ C

tensor([ 0.3785, -1.0079])

In [7]:
C[[5, 6, 7]]

tensor([[ 0.3785, -1.0079],
        [ 0.1047,  1.1572],
        [-1.7965,  0.2379]])

In [10]:
emb = C[X]
emb.shape

torch.Size([32, 3, 2])

## **Hidden Layer**

In [11]:
W1 = torch.randn((6,100))
b1 = torch.randn(100)

In [ ]:
# torch.cat([emb[:, 0, :], emb[:, 1, :], emb[:, 2, :]], 1)
# torch.cat(torch.unbind(emb, 1), 1)

In [15]:
a = torch.arange(18)
a

tensor([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17])

In [ ]:
# PyTorch doesn't actually copy or move the underlying data in memory, it just changes the viewfinder
a.view(2, 9)

tensor([[ 0,  1,  2,  3,  4,  5,  6,  7,  8],
        [ 9, 10, 11, 12, 13, 14, 15, 16, 17]])

In [ ]:
emb.view(32, 6)

In [ ]:
emb.view(emb.shape[-1], 6) @ W1 + b1

tensor([[ -1.7053,   0.0906, -10.6945,  ...,  -0.3534,  -3.0828,  -2.5576],
        [  0.9136,  -0.2992,  -6.2221,  ...,   0.5998,  -1.5680,  -4.7199],
        [ -0.4684,   1.0951,  -7.0582,  ...,  -1.5842,  -4.9020,  -3.8835],
        ...,
        [ -3.6557,   0.6108,  -0.2701,  ...,   0.8442,   0.8297,  -0.1932],
        [ -0.6082,  -0.8469,  -2.2952,  ...,   1.4368,   2.0772,  -0.5968],
        [ -3.1810,   0.1049,  -8.2538,  ...,   0.6081,  -1.3595,  -3.7842]])